
R.O.M.  —  MASS / G - FREE CLOSURE DEMONSTRATION
================================================

Claim under test (structural, NOT "ROM vs GR correctness"):
  The R.O.M. closed algebraic system determines the full orbital geometry from
  OBSERVABLE inputs alone.  Newton's constant G and the mass M are never needed
  as inputs; they appear only at the very end as optional unit conversions
  (a length $R_s$ -> a mass in kg).  No metric, curvature, geodesic, action
  principle, perturbation series, or differential equation is used.

How the demonstration is kept honest:
  1. A "truth" system is built from (M, a, e) in SI units and its observables
     are generated with STANDARD physics, independent of ROM:
        - Keplerian ellipse           $r(O) = a(1-e^2)/(1+e \cos(O))$
        - vis-viva                    $v^2  = GM(2/r - 1/a)$
        - Kepler third law            $T    = 2*\pi*\sqrt{a^3/GM}$
        - GR gravitational redshift   $1+z_k = 1/\sqrt{1-R_s/r}$
        - SR transverse Doppler       $1+z_b = 1/\sqrt{1-v^2/c^2}$
  2. Only OBSERVABLES are handed to the ROM closure routines:
        $\beta$ (transverse-Doppler projection), $e$, period $T$,  and/or  
        $z_{\kappa}$, $z_{\beta}$.
     The closure routines have NO G and NO M in scope (enforced below).
  3. ROM reconstructs $a$, $R_s$, $r_p$, $r_a$ and the whole phase-resolved orbit, and
     recovers GM (a length^3/time^2 combination, no G) and finally M (kg, G used
     ONLY here as a conversion).  Everything is checked against the truth.

In [1]:
import mpmath as mp
mp.mp.dps = 30
pi = mp.pi
sqrt, sin, cos = mp.sqrt, mp.sin, mp.cos

# ======================================================================
# 0.  Physical constants  (SI)  — used ONLY by the truth generator and by
#     the final optional "kg conversion", never inside ROM closure.
# ======================================================================
G    = mp.mpf('6.67430e-11')      # m^3 kg^-1 s^-2
c    = mp.mpf('2.99792458e8')     # m/s
Msun = mp.mpf('1.98892e30')       # kg

# ======================================================================
# 1.  TRUTH  (standard physics).  Fundamental truth = (M_true, a_true, e).
# ======================================================================
M_true   = mp.mpf('1.4')*Msun
beta_aim = mp.mpf('0.01')                          # target equilibrium beta
a_true   = G*M_true/(c**2*beta_aim**2)             # gives beta_glob = 0.01
e        = mp.mpf('0.88')

Rs_true   = 2*G*M_true/c**2
beta_glob = sqrt(G*M_true/(c**2*a_true))           # global kinetic projection
kappa_tru = sqrt(Rs_true/a_true)
T_true    = 2*pi*sqrt(a_true**3/(G*M_true))        # Kepler III
rp_true   = a_true*(1-e)
ra_true   = a_true*(1+e)

# global-projection redshifts (what the closure's redshift-mode consumes)
kappaX_t = sqrt(1-kappa_tru**2)
betaY_t  = sqrt(1-beta_glob**2)
z_kappa_glob = 1/kappaX_t - 1
z_beta_glob  = 1/betaY_t  - 1

def truth_phase(O):
    """Independent standard-physics state at true anomaly O."""
    r  = a_true*(1-e**2)/(1+e*cos(O))
    v2 = G*M_true*(2/r - 1/a_true)                 # vis-viva
    h  = sqrt(G*M_true*a_true*(1-e**2))            # specific ang. momentum
    vT = h/r
    vR = sqrt(v2 - vT**2) * (1 if sin(O) >= 0 else -1)
    kappa_o = sqrt(Rs_true/r)
    beta_o  = sqrt(v2)/c
    zk = 1/sqrt(1-kappa_o**2) - 1
    zb = 1/sqrt(1-beta_o**2)  - 1
    return dict(r=r, beta_o=beta_o, beta_T=vT/c, beta_R=vR/c,
                kappa_o=kappa_o, z_kappa=zk, z_beta=zb,
                Zsys=(1+zk)*(1+zb))

# OBSERVABLES exposed to ROM (NOTHING about M, G, a, R_s is exposed):
OBS = dict(beta=beta_glob, e=e, T=T_true,
           z_kappa=z_kappa_glob, z_beta=z_beta_glob)

# ======================================================================
# 2.  ROM CLOSURE  — functions take observables + c only.  No G, no M.
# ======================================================================
def rom_from_kinematics(beta, e, T, c):
    """Inputs: beta (dimensionless), e (dimensionless), T (s), c (m/s)."""
    kappa = sqrt(2)*beta                      # closure  kappa^2 = 2 beta^2
    a     = T*beta*c/(2*pi)                    # <-- scale WITHOUT GM
    Rs    = kappa**2 * a                       # = 2 beta^2 a   (a length)
    out = dict(beta=beta, kappa=kappa, a=a, Rs=Rs,
               rp=a*(1-e), ra=a*(1+e),
               kappaX=sqrt(1-kappa**2), betaY=sqrt(1-beta**2))
    out['tau']  = out['kappaX']*out['betaY']
    out['Zsys'] = 1/out['tau']
    out['Q']    = sqrt(kappa**2 + beta**2)
    out['GM']   = beta**2 * c**2 * a           # G*M recovered as a COMBINATION (no G)
    return out

def rom_from_redshifts(z_kappa, z_beta, e, T, c):
    """Inputs: two redshifts + e + T + c.  No G, no M."""
    beta  = sqrt(1 - 1/(1+z_beta)**2)          # transverse Doppler  -> beta
    kappa = sqrt(1 - 1/(1+z_kappa)**2)         # grav. redshift      -> kappa
    closure_residual = kappa**2 - 2*beta**2    # DATA consistency test (expect 0)
    a   = T*beta*c/(2*pi)
    Rs_from_kappa = kappa**2 * a
    Rs_from_beta  = 2*beta**2 * a              # independent route; must agree
    return dict(beta=beta, kappa=kappa, closure_residual=closure_residual,
                a=a, Rs_from_kappa=Rs_from_kappa, Rs_from_beta=Rs_from_beta,
                rp=a*(1-e), ra=a*(1+e))

def rom_phase(beta, e, a, Rs, O):
    """Phase-resolved state from ROM, using only observation-derived a, Rs."""
    r       = a*(1-e**2)/(1+e*cos(O))                       # = Rs/kappa_o^2
    kappa_o = sqrt(Rs/r)
    beta_o  = sqrt(Rs/r - Rs/(2*a))                         # vis-viva, ROM form
    beta_T  = beta*(1+e*cos(O))/sqrt(1-e**2)
    beta_R  = beta*e*sin(O)/sqrt(1-e**2)
    zk = 1/sqrt(1-kappa_o**2) - 1
    zb = 1/sqrt(1-beta_o**2)  - 1
    return dict(r=r, beta_o=beta_o, beta_T=beta_T, beta_R=beta_R,
                kappa_o=kappa_o, z_kappa=zk, z_beta=zb, Zsys=(1+zk)*(1+zb))

# ======================================================================
# 3.  RUN CLOSURE FROM OBSERVABLES + VERIFY against truth
# ======================================================================
def rel(x, y):
    return abs(x-y)/max(abs(y), mp.mpf(1))

print("="*78)
print("R.O.M.  MASS/G-FREE CLOSURE")
print("="*78)
print("Truth (hidden from closure): M = 1.4 Msun, a = %.6e m, e = %.2f" % (float(a_true), float(e)))
print("Observable inputs handed to ROM: beta=%.6f, e=%.2f, T=%.6f s" % (float(OBS['beta']), float(OBS['e']), float(OBS['T'])))
print("                                 z_kappa=%.6e, z_beta=%.6e" % (float(OBS['z_kappa']), float(OBS['z_beta'])))

R = rom_from_kinematics(OBS['beta'], OBS['e'], OBS['T'], c)

print("\n--- Geometry recovered from {beta, e, T} alone (NO G, NO M) ---")
rows = [("a   (semi-major axis)", R['a'], a_true),
        ("R_s (Schwarzschild radius)", R['Rs'], Rs_true),
        ("r_p (periapsis)", R['rp'], rp_true),
        ("r_a (apoapsis)", R['ra'], ra_true),
        ("kappa", R['kappa'], kappa_tru)]
allok = True
for name, got, tru in rows:
    ok = rel(got, tru) < mp.mpf('1e-25'); allok &= ok
    print("   %-26s ROM=%.10e  truth=%.10e  rel=%.1e  %s"
          % (name, float(got), float(tru), float(rel(got, tru)), "OK" if ok else "**FAIL**"))

print("\n--- Mass parameter as an OUTPUT (this is where G, M finally appear) ---")
GM_true = G*M_true
ok_gm = rel(R['GM'], GM_true) < mp.mpf('1e-25'); allok &= ok_gm
print("   GM = beta^2 c^2 a     ROM=%.10e  truth=%.10e  rel=%.1e  %s   (NO G used)"
      % (float(R['GM']), float(GM_true), float(rel(R['GM'], GM_true)), "OK" if ok_gm else "**FAIL**"))
M_kg = R['Rs']*c**2/(2*G)          # <-- the ONLY place G is used: length -> kg
ok_m = rel(M_kg, M_true) < mp.mpf('1e-25'); allok &= ok_m
print("   M  = R_s c^2 /(2G)    ROM=%.10e  truth=%.10e  rel=%.1e  %s   (G = unit conversion)"
      % (float(M_kg), float(M_true), float(rel(M_kg, M_true)), "OK" if ok_m else "**FAIL**"))
print("   M  in solar masses:   ROM=%.8f  truth=%.8f" % (float(M_kg/Msun), float(M_true/Msun)))

# ----- contrast with the classical route -----
print("\n--- Why this is non-trivial: classical scale needs GM, ROM's does not ---")
a_classical = (G*M_true*T_true**2/(4*pi**2))**(mp.mpf(1)/3)   # Kepler: requires GM as INPUT
print("   classical a = (G M T^2/4pi^2)^(1/3) = %.6e m   <- REQUIRES GM as input" % float(a_classical))
print("   ROM      a = T beta c /(2 pi)       = %.6e m   <- requires only T, beta" % float(R['a']))

# ======================================================================
# 4.  Redshift-mode closure + on-data closure test
# ======================================================================
print("\n" + "="*78)
print("CLOSURE FROM REDSHIFTS {z_kappa, z_beta, e, T}  (+ on-data closure test)")
print("="*78)
Rz = rom_from_redshifts(OBS['z_kappa'], OBS['z_beta'], OBS['e'], OBS['T'], c)
print("   beta  recovered = %.12f   (truth %.12f)" % (float(Rz['beta']), float(beta_glob)))
print("   kappa recovered = %.12f   (truth %.12f)" % (float(Rz['kappa']), float(kappa_tru)))
print("   closure test  kappa^2 - 2 beta^2 = %.2e   (expect 0 if data obeys ROM)" % float(Rz['closure_residual']))
print("   R_s via kappa = %.10e" % float(Rz['Rs_from_kappa']))
print("   R_s via beta  = %.10e   (two independent routes agree: %s)"
      % (float(Rz['Rs_from_beta']),
         rel(Rz['Rs_from_kappa'], Rz['Rs_from_beta']) < mp.mpf('1e-25')))

# ======================================================================
# 5.  Input-symmetry: several different observable sets -> same system
# ======================================================================
print("\n" + "="*78)
print("INPUT-SYMMETRY: different observable sets close to the SAME geometry")
print("="*78)
# Set A already done: {beta, e, T}
# Set C: {R_s, beta, e}  (someone measured R_s and beta)   -> a, T
a_C  = R['Rs']/(2*OBS['beta']**2)
T_C  = 2*pi*a_C/(OBS['beta']*c)
# Set D: {a, e, T}  -> beta, Rs
beta_D = 2*pi*R['a']/(T_true*c)
Rs_D   = 2*beta_D**2*R['a']
print("   Set A {beta,e,T} : a=%.6e  Rs=%.6e" % (float(R['a']), float(R['Rs'])))
print("   Set C {Rs,beta,e}: a=%.6e  T=%.6f s  (T matches: %s)"
      % (float(a_C), float(T_C), rel(T_C, T_true) < mp.mpf('1e-25')))
print("   Set D {a,e,T}    : beta=%.8f  Rs=%.6e  (beta,Rs match: %s)"
      % (float(beta_D), float(Rs_D),
         (rel(beta_D, beta_glob) < mp.mpf('1e-25') and rel(Rs_D, R['Rs']) < mp.mpf('1e-25'))))

# ======================================================================
# 6.  Phase-resolved orbit reconstructed from observables vs truth
# ======================================================================
print("\n" + "="*78)
print("PHASE-RESOLVED ORBIT  (ROM from observables)  vs  TRUTH (standard physics)")
print("="*78)
print("   O[deg]     r/a        beta_o(ROM/tru)        z_beta(ROM/tru)     ok")
phase_ok = True
for deg in [0, 30, 60, 90, 150, 210, 300]:
    O = mp.mpf(deg)*pi/180
    P = rom_phase(OBS['beta'], OBS['e'], R['a'], R['Rs'], O)
    Tr = truth_phase(O)
    ok = (rel(P['beta_o'], Tr['beta_o']) < mp.mpf('1e-22')
          and rel(P['z_beta'], Tr['z_beta']) < mp.mpf('1e-22')
          and rel(P['r'], Tr['r']) < mp.mpf('1e-22')
          and rel(P['kappa_o'], Tr['kappa_o']) < mp.mpf('1e-22'))
    phase_ok &= ok
    print("   %5d   %8.5f   %.10f/%.10f   %.4e/%.4e  %s"
          % (deg, float(P['r']/R['a']), float(P['beta_o']), float(Tr['beta_o']),
             float(P['z_beta']), float(Tr['z_beta']), "OK" if ok else "FAIL"))

# ======================================================================
# 7.  Ledger
# ======================================================================
print("\n" + "="*78)
print("LEDGER")
print("="*78)
print("  Inputs actually used by closure : beta (or z_beta), kappa (or z_kappa), e, T, c")
print("  Quantities that are OUTPUTS     : a, R_s, r_p, r_a, full orbit, GM, and M")
print("  Where G appears                 : ONLY in M = R_s c^2/(2G)  (length -> kg)")
print("  Where M appears                 : ONLY as that final output")
print("  GM itself                       : = beta^2 c^2 a  (no G needed at all)")
print("  Differential equations used     : NONE")
print("  Metric / curvature / geodesic   : NONE")
print("\n  OVERALL: %s" % ("ALL CHECKS PASS" if (allok and phase_ok) else "SOME CHECKS FAILED"))

R.O.M.  MASS/G-FREE CLOSURE
Truth (hidden from closure): M = 1.4 Msun, a = 2.067805e+07 m, e = 0.88
Observable inputs handed to ROM: beta=0.010000, e=0.88, T=43.337997 s
                                 z_kappa=1.000150e-04, z_beta=5.000375e-05

--- Geometry recovered from {beta, e, T} alone (NO G, NO M) ---
   a   (semi-major axis)      ROM=2.0678054155e+07  truth=2.0678054155e+07  rel=1.6e-31  OK
   R_s (Schwarzschild radius) ROM=4.1356108311e+03  truth=4.1356108311e+03  rel=0.0e+00  OK
   r_p (periapsis)            ROM=2.4813664987e+06  truth=2.4813664987e+06  rel=1.7e-31  OK
   r_a (apoapsis)             ROM=3.8874741812e+07  truth=3.8874741812e+07  rel=1.7e-31  OK
   kappa                      ROM=1.4142135624e-02  truth=1.4142135624e-02  rel=0.0e+00  OK

--- Mass parameter as an OUTPUT (this is where G, M finally appear) ---
   GM = beta^2 c^2 a     ROM=1.8584508258e+20  truth=1.8584508258e+20  rel=1.6e-31  OK   (NO G used)
   M  = R_s c^2 /(2G)    ROM=2.7844880000e+30  truth=2.7